In [ ]:
!pip uninstall -y cupy cupy-cuda11x cupy-cuda12x cupy-cuda13x
!pip install -q cupy-cuda12x

Found existing installation: cupy-cuda13x 14.2.0
Uninstalling cupy-cuda13x-14.2.0:
  Successfully uninstalled cupy-cuda13x-14.2.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 MB 7.6 MB/s eta 0:00:00


In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download('treebank')

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

In [ ]:
from nltk.corpus import treebank

sentences = treebank.sents()

print("Number of sentences:", len(sentences))
print("First sentence:")
print(sentences[10])
parsed_sentences = treebank.parsed_sents()

Number of sentences: 3914
First sentence:
['Neither', 'Lorillard', 'nor', 'the', 'researchers', 'who', '*T*-3', 'studied', 'the', 'workers', 'were', 'aware', 'of', 'any', 'research', 'on', 'smokers', 'of', 'the', 'Kent', 'cigarettes', '.']


In [ ]:
# #Adding BOS and EOS tags
for sentence in range(len(sentences)):
  sentences[sentence].insert(0,"<BOS>")
  sentences[sentence].append("<EOS>")


In [ ]:
X=[]
Y=[]
for sentence in sentences:
  length=len(sentence)
  X.append(sentence[:length-1])
  Y.append(sentence[1:length])

In [ ]:
from collections import Counter

# Count all words in the corpus
counter = Counter()

for sentence in sentences:
    counter.update(sentence)

# Create vocabulary
vocab = sorted(counter.keys())

# Word → integer
word_to_idx = {
    word: idx for idx, word in enumerate(vocab)
}

# Integer → word
idx_to_word = {
    idx: word for word, idx in word_to_idx.items()
}

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)

Vocabulary size: 12410


In [ ]:
X_encoded = []
Y_encoded = []

for x_sentence, y_sentence in zip(X, Y):

    X_encoded.append([
        word_to_idx[word]
        for word in x_sentence
    ])

    Y_encoded.append([
        word_to_idx[word]
        for word in y_sentence
    ])

In [ ]:
import random

# Keep each X-Y sentence pair together
pairs = list(zip(X_encoded, Y_encoded))

# Shuffle reproducibly
random.seed(42)
random.shuffle(pairs)

# 80% train, 10% validation, 10% test
n = len(pairs)

train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_pairs = pairs[:train_end]
val_pairs = pairs[train_end:val_end]
test_pairs = pairs[val_end:]

# Separate X and Y again
X_train, Y_train = zip(*train_pairs)
X_val, Y_val = zip(*val_pairs)
X_test, Y_test = zip(*test_pairs)

# Convert back to lists
X_train = list(X_train)
Y_train = list(Y_train)

X_val = list(X_val)
Y_val = list(Y_val)

X_test = list(X_test)
Y_test = list(Y_test)

In [ ]:
import cupy as cp


class RNNLanguageModel:

    def __init__(self, vocab_size, embedding_size, hidden_size):

        self.vocab_size = vocab_size
        self.embedding_size = embedding_size
        self.hidden_size = hidden_size

        # ------------------------------------------------
        # Embedding layer
        # ------------------------------------------------
        self.E = cp.random.randn(vocab_size, embedding_size) * 0.01

        # ------------------------------------------------
        # RNN Layer 1
        # ------------------------------------------------
        self.W_xh1 = cp.random.randn(hidden_size, embedding_size) * 0.01
        self.W_hh1 = cp.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h1 = cp.zeros((hidden_size, 1))

        # ------------------------------------------------
        # RNN Layer 2
        # ------------------------------------------------
        self.W_xh2 = cp.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hh2 = cp.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h2 = cp.zeros((hidden_size, 1))

        # ------------------------------------------------
        # RNN Layer 3
        # ------------------------------------------------
        self.W_xh3 = cp.random.randn(hidden_size, hidden_size) * 0.01
        self.W_hh3 = cp.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h3 = cp.zeros((hidden_size, 1))

        # ------------------------------------------------
        # Output layer
        # ------------------------------------------------
        self.W_hy = cp.random.randn(vocab_size, hidden_size) * 0.01
        self.b_y = cp.zeros((vocab_size, 1))

    # ==================================================
    # Forward pass
    # ==================================================
    def forward(self, x):
        outputs = []
        x_emb = []
        for i in x:
            x_emb.append(self.E[i])

        h1_prev = cp.zeros((self.hidden_size, 1))
        h2_prev = cp.zeros((self.hidden_size, 1))
        h3_prev = cp.zeros((self.hidden_size, 1))
        h1 = [h1_prev]
        h2 = [h2_prev]
        h3 = [h3_prev]

        for input in x_emb:
            # reshape embedding row -> column vector
            input_col = input.reshape(-1, 1)

            # Hidden Layer 1
            input_transformation = self.W_xh1 @ input_col
            hidden_transformation = self.W_hh1 @ h1[-1]
            h1_t = cp.tanh(input_transformation + hidden_transformation + self.b_h1)
            h1.append(h1_t)

            # Hidden Layer 2
            input_transformation_2 = self.W_xh2 @ h1_t
            hidden_transformation_2 = self.W_hh2 @ h2[-1]
            h2_t = cp.tanh(input_transformation_2 + hidden_transformation_2 + self.b_h2)
            h2.append(h2_t)

            # Hidden Layer 3
            input_transformation_3 = self.W_xh3 @ h2_t
            hidden_transformation_3 = self.W_hh3 @ h3[-1]
            h3_t = cp.tanh(input_transformation_3 + hidden_transformation_3 + self.b_h3)
            h3.append(h3_t)

            # Output
            output = self.W_hy @ h3_t + self.b_y

            # Softmax
            shifted_output = output - cp.max(output)
            exp_output = cp.exp(shifted_output)
            y_cap_t = exp_output / cp.sum(exp_output)

            outputs.append(y_cap_t)

        # IMPORTANT: return happens AFTER the loop, for every timestep
        return outputs, h1, h2, h3, x_emb

    # ==================================================
    # Cross entropy (utility, not used internally)
    # ==================================================
    @staticmethod
    def cross_entropy(y_pred, target):
        epsilon = 1e-12
        probability = y_pred[target, 0]
        return -cp.log(probability + epsilon)

    # ==================================================
    # Backward pass (full BPTT)
    # ==================================================
    def backward(self, x, target, h1, h2, h3, outputs, x_emb):
        T = len(outputs)

        dW_hy = cp.zeros_like(self.W_hy)
        db_y = cp.zeros_like(self.b_y)

        dW_xh3 = cp.zeros_like(self.W_xh3)
        dW_hh3 = cp.zeros_like(self.W_hh3)
        db_h3 = cp.zeros_like(self.b_h3)

        dW_xh2 = cp.zeros_like(self.W_xh2)
        dW_hh2 = cp.zeros_like(self.W_hh2)
        db_h2 = cp.zeros_like(self.b_h2)

        dW_xh1 = cp.zeros_like(self.W_xh1)
        dW_hh1 = cp.zeros_like(self.W_hh1)
        db_h1 = cp.zeros_like(self.b_h1)

        dE = cp.zeros_like(self.E)

        # Gradients carried backward through time
        dh3_next = cp.zeros((self.hidden_size, 1))
        dh2_next = cp.zeros((self.hidden_size, 1))
        dh1_next = cp.zeros((self.hidden_size, 1))

        for t in reversed(range(T)):

            # -------------------------
            # Output layer
            # -------------------------
            dy = outputs[t].copy()
            dy[target[t]] -= 1

            dW_hy += dy @ h3[t + 1].T
            db_y += dy

            dh3 = self.W_hy.T @ dy
            dh3 += dh3_next  # gradient from future timestep

            # -------------------------
            # RNN Layer 3
            # -------------------------
            da3 = dh3 * (1 - h3[t + 1] ** 2)

            dW_xh3 += da3 @ h2[t + 1].T
            dW_hh3 += da3 @ h3[t].T
            db_h3 += da3

            dh2 = self.W_xh3.T @ da3
            dh3_next = self.W_hh3.T @ da3

            # -------------------------
            # RNN Layer 2
            # -------------------------
            dh2 += dh2_next

            da2 = dh2 * (1 - h2[t + 1] ** 2)

            dW_xh2 += da2 @ h1[t + 1].T
            dW_hh2 += da2 @ h2[t].T
            db_h2 += da2

            dh1 = self.W_xh2.T @ da2
            dh2_next = self.W_hh2.T @ da2

            # -------------------------
            # RNN Layer 1
            # -------------------------
            dh1 += dh1_next

            da1 = dh1 * (1 - h1[t + 1] ** 2)

            dW_xh1 += da1 @ x_emb[t].reshape(1, -1)
            dW_hh1 += da1 @ h1[t].T
            db_h1 += da1

            dh1_next = self.W_hh1.T @ da1

            # -------------------------
            # Embedding gradient
            # -------------------------
            dx = self.W_xh1.T @ da1          # (embedding_size, 1)
            dE[x[t]] += dx.flatten()

        # IMPORTANT: return happens AFTER the loop, once all timesteps
        # have been accumulated
        grads = {
            "dE": dE,
            "dW_xh1": dW_xh1, "dW_hh1": dW_hh1, "db_h1": db_h1,
            "dW_xh2": dW_xh2, "dW_hh2": dW_hh2, "db_h2": db_h2,
            "dW_xh3": dW_xh3, "dW_hh3": dW_hh3, "db_h3": db_h3,
            "dW_hy": dW_hy, "db_y": db_y,
        }

        self._clip_grads(grads, max_norm=5.0)
        return grads

    # ==================================================
    # Gradient clipping (global norm) — essential for a
    # 3-layer stacked tanh RNN trained with plain SGD
    # ==================================================
    @staticmethod
    def _clip_grads(grads, max_norm=5.0):
        total_norm_sq = 0.0
        for g in grads.values():
            total_norm_sq += float(cp.sum(g ** 2))
        total_norm = total_norm_sq ** 0.5

        if total_norm > max_norm:
            scale = max_norm / (total_norm + 1e-6)
            for k in grads:
                grads[k] *= scale
    # ==================================================
    # Evaluation (forward pass only — no backprop, no updates)
    # ==================================================
    def evaluate(self, X, Y):
        total_loss = cp.float32(0.0)
        total_tokens = 0

        for x, target in zip(X, Y):
            outputs, h1, h2, h3, x_emb = self.forward(x)
            for t in range(len(target)):
                total_loss += -cp.log(outputs[t][target[t], 0] + 1e-12)
            total_tokens += len(target)

        return float(total_loss.get()) / total_tokens

    # ==================================================
    # Parameter update (vanilla SGD)
    # ==================================================
    def update_parameters(self, grads, learning_rate):
        self.E -= learning_rate * grads["dE"]

        self.W_xh1 -= learning_rate * grads["dW_xh1"]
        self.W_hh1 -= learning_rate * grads["dW_hh1"]
        self.b_h1 -= learning_rate * grads["db_h1"]

        self.W_xh2 -= learning_rate * grads["dW_xh2"]
        self.W_hh2 -= learning_rate * grads["dW_hh2"]
        self.b_h2 -= learning_rate * grads["db_h2"]

        self.W_xh3 -= learning_rate * grads["dW_xh3"]
        self.W_hh3 -= learning_rate * grads["dW_hh3"]
        self.b_h3 -= learning_rate * grads["db_h3"]

        self.W_hy -= learning_rate * grads["dW_hy"]
        self.b_y -= learning_rate * grads["db_y"]

    # ==================================================
    # Training loop
    # ==================================================
    def train(self, X_train, Y_train, epochs=10, learning_rate=0.001,
              X_val=None, Y_val=None):

        n = len(X_train)
        indices = list(range(n))

        for epoch in range(epochs):

            cp.random.shuffle(cp.array(indices))  # reshuffle order each epoch
            total_loss = cp.float32(0.0)
            total_tokens = 0

            for idx in indices:
                x, target = X_train[idx], Y_train[idx]

                # Forward pass
                outputs, h1, h2, h3, x_emb = self.forward(x)

                # Loss (kept on-device, no per-sample .get())
                for t in range(len(target)):
                    total_loss += -cp.log(outputs[t][target[t], 0] + 1e-12)
                total_tokens += len(target)

                # Backward pass
                grads = self.backward(x, target, h1, h2, h3, outputs, x_emb)

                # Update parameters
                self.update_parameters(grads, learning_rate)

            average_loss = float(total_loss.get()) / total_tokens

            if X_val is not None and Y_val is not None:
                val_loss = self.evaluate(X_val, Y_val)
                print(
                    f"Epoch {epoch + 1}/{epochs} "
                    f"- Loss: {average_loss:.4f} - Val Loss: {val_loss:.4f}"
                )
            else:
                print(f"Epoch {epoch + 1}/{epochs} - Loss: {average_loss:.4f}")

In [ ]:
def train_test_split(X, Y, test_size=0.2, seed=None, shuffle=True):
    """
    Split parallel lists of sequences (X, Y) into train/test sets.

    Works with variable-length sequences (list of lists / list of arrays),
    unlike numpy/cupy slicing which assumes a fixed-size array.

    Parameters
    ----------
    X, Y : list
        Parallel lists — X[i] is the input sequence, Y[i] is its target
        sequence. Must be the same length.
    test_size : float or int
        Fraction (0 < test_size < 1) or absolute count of samples to
        hold out for testing.
    seed : int, optional
        Random seed for reproducible splits.
    shuffle : bool
        Whether to shuffle before splitting. Almost always want True,
        unless your data has a meaningful order you want to preserve
        (e.g. chronological).

    Returns
    -------
    X_train, X_test, Y_train, Y_test : lists
    """
    if len(X) != len(Y):
        raise ValueError(
            f"X and Y must be the same length, got {len(X)} and {len(Y)}"
        )

    n = len(X)
    indices = list(range(n))

    if shuffle:
        rng = random.Random(seed)
        rng.shuffle(indices)

    if isinstance(test_size, float):
        if not 0.0 < test_size < 1.0:
            raise ValueError("test_size as a float must be between 0 and 1")
        n_test = int(round(n * test_size))
    else:
        n_test = int(test_size)
        if not 0 <= n_test <= n:
            raise ValueError(f"test_size as an int must be between 0 and {n}")

    test_indices = indices[:n_test]
    train_indices = indices[n_test:]

    X_train = [X[i] for i in train_indices]
    Y_train = [Y[i] for i in train_indices]
    X_test = [X[i] for i in test_indices]
    Y_test = [Y[i] for i in test_indices]

    return X_train, X_test, Y_train, Y_test

In [ ]:
# X_train, X_test, Y_train, Y_test = train_test_split(
#     X, Y, test_size=0.2, seed=42
# )
# X_train[0]

model = RNNLanguageModel(vocab_size=len(vocab), embedding_size=128, hidden_size=128)
model.train(
    X_train, Y_train,
    epochs=50,
    learning_rate=0.001,
    X_val=X_test, Y_val=Y_test,   # tracks generalization each epoch
)

test_loss = model.evaluate(X_test, Y_test)

Epoch 1/50 - Loss: 8.6450 - Val Loss: 7.7448
Epoch 2/50 - Loss: 7.5157 - Val Loss: 7.4036
Epoch 3/50 - Loss: 7.3126 - Val Loss: 7.2872
Epoch 4/50 - Loss: 7.2205 - Val Loss: 7.2220
Epoch 5/50 - Loss: 7.1616 - Val Loss: 7.1775
Epoch 6/50 - Loss: 7.1188 - Val Loss: 7.1447
Epoch 7/50 - Loss: 7.0863 - Val Loss: 7.1200
Epoch 8/50 - Loss: 7.0609 - Val Loss: 7.1009
Epoch 9/50 - Loss: 7.0405 - Val Loss: 7.0858
Epoch 10/50 - Loss: 7.0238 - Val Loss: 7.0737
Epoch 11/50 - Loss: 7.0098 - Val Loss: 7.0636
Epoch 12/50 - Loss: 6.9979 - Val Loss: 7.0552
Epoch 13/50 - Loss: 6.9876 - Val Loss: 7.0480
Epoch 14/50 - Loss: 6.9785 - Val Loss: 7.0419
Epoch 15/50 - Loss: 6.9705 - Val Loss: 7.0365
Epoch 16/50 - Loss: 6.9634 - Val Loss: 7.0318
Epoch 17/50 - Loss: 6.9570 - Val Loss: 7.0277
Epoch 18/50 - Loss: 6.9511 - Val Loss: 7.0241
Epoch 19/50 - Loss: 6.9459 - Val Loss: 7.0209
Epoch 20/50 - Loss: 6.9411 - Val Loss: 7.0181
Epoch 21/50 - Loss: 6.9368 - Val Loss: 7.0157
Epoch 22/50 - Loss: 6.9328 - Val Loss: 7.01